# Implementation audit

<a href="https://colab.research.google.com/github/kootru-repo/charge-filtered-cumulant-residuals/blob/main/notebooks/03_implementation_audit.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" width="200"/></a>


Manuscript reference: Section VI.

The full audit covers $3679$ charge-neutral word observables across $26$ fixed-$N$ states with $n \in \{4, 6, 8\}$ and word lengths $m \in \{3, 4, 5\}$. This notebook does two things:

1. Loads the deposited screen JSONs and verifies the headline counts and the empirical effective constant $B^{\mathrm{eff}}_{\max} \approx 2.0$.
2. Runs a representative smoke audit on a single $n = 4$ Slater-determinant cell and confirms the bound is respected on every catalog observable.

**What this notebook verifies vs. does not verify.** Cells that load deposited JSON files and read out headline numbers (Sections "Headline counts" and "$B^{\mathrm{eff}}_{\max}$") verify the **archive**, not the underlying computation: they confirm the values in `data/*.json` match what the manuscript reports. The **science** (that those values are correct outputs of the pipeline) is verified by:

- `tests/test_smoke_regeneration.py`, which regenerates the Hubbard $U=0$ baseline from code and compares to the deposited file bit-exactly modulo volatile metadata;
- the "Smoke audit on a tiny cell" section below, which runs the catalog evaluation pipeline live on a single $n=4$ cell;
- the full audit, which is deterministically regenerable from code locally.

Together with the SHA256 checks in `tests/test_data_integrity.py`, the archive-readback + smoke-regeneration pair gives end-to-end verification.

In [ ]:
# Bootstrap on Colab / fresh env (skipped if package + data are already present).
# Local users who cloned the repo and ran 'uv sync' skip the install entirely.
# On Colab the bootstrap clones the repo so data/ files are reachable via
# notebook-relative paths, chdir's into it, then installs the package with uv.
import importlib, importlib.util as _ilu
import pathlib as _pl
_HAS_PKG = _ilu.find_spec("connected_layer_sector") is not None
_HAS_DATA = _pl.Path("data").is_dir() or _pl.Path("../data").is_dir()
if not (_HAS_PKG and _HAS_DATA):
    import subprocess as _sp, os as _os
    _REPO = "/content/charge-filtered-cumulant-residuals"
    if not _pl.Path(_REPO).is_dir():
        _sp.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/kootru-repo/charge-filtered-cumulant-residuals.git",
            _REPO,
        ])
    _os.chdir(_REPO)
    # Pin uv to a known directory. UV_INSTALL_DIR is the installer's explicit
    # override; without it the location depends on CARGO_HOME / XDG_BIN_HOME
    # inheritance (Colab differs from local Linux).
    _UV_DIR = "/tmp/uv-bootstrap"
    _os.makedirs(_UV_DIR, exist_ok=True)
    _env = {**_os.environ, "UV_INSTALL_DIR": _UV_DIR}
    _sp.check_call(
        "curl -LsSf https://astral.sh/uv/install.sh | sh",
        shell=True, executable="/bin/bash", env=_env,
    )
    _UV = f"{_UV_DIR}/uv"
    # Non-editable install: lands directly in site-packages (already on sys.path),
    # so the import below works without sys.path gymnastics. Editable installs
    # rely on .pth files Python only reads at startup.
    _sp.check_call([_UV, "pip", "install", "--system", "-q", "."])
    importlib.invalidate_caches()  # flush find_spec cache so re-runs idempotently short-circuit

import connected_layer_sector
print(f"connected_layer_sector {connected_layer_sector.__version__} ready")


In [1]:
import json
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"

for fn in ["screen_sector_cumulant_theorem.json",
          "screen_sector_cumulant_extended.json",
          "screen_sector_audit_r5.json",
          "audit_sector_cumulant_results.json"]:
    p = DATA / fn
    print(f"  {fn}: {'exists' if p.exists() else 'missing'}  ({p.stat().st_size if p.exists() else 0} bytes)")

  screen_sector_cumulant_theorem.json: exists  (3188 bytes)
  screen_sector_cumulant_extended.json: exists  (5613 bytes)
  screen_sector_audit_r5.json: exists  (4857 bytes)
  audit_sector_cumulant_results.json: exists  (2698 bytes)


## Headline counts from the deposited JSONs

Manuscript Sec VI: $3679$ observables = $630 + 1960 + 1089$ across the three audit screens. $26$ fixed-$N$ states = $6 + 11 + 9$.

In [ ]:
screens = [
    "screen_sector_cumulant_theorem.json",
    "screen_sector_cumulant_extended.json",
    "screen_sector_audit_r5.json",
]

expected_per_screen = {
    "screen_sector_cumulant_theorem.json":  (6, 630),
    "screen_sector_cumulant_extended.json": (11, 1960),
    "screen_sector_audit_r5.json":          (9, 1089),
}

total_obs = 0
total_cells = 0
for fn in screens:
    p = DATA / fn
    j = json.loads(p.read_text(encoding="utf-8"))
    cells = j.get("cells") or j.get("rows") or []
    n_obs = sum(c.get("n_obs", 0) for c in cells)
    exp_cells, exp_obs = expected_per_screen[fn]
    print(f"  {fn}: {len(cells)} cells, {n_obs} observables  (manuscript: {exp_cells}, {exp_obs})")
    assert len(cells) == exp_cells, f"{fn}: {len(cells)} cells != manuscript {exp_cells}"
    assert n_obs == exp_obs, f"{fn}: {n_obs} observables != manuscript {exp_obs}"
    total_obs += n_obs
    total_cells += len(cells)

print()
print(f"Total observables across screens: {total_obs}  (manuscript: 3679)")
print(f"Total fixed-N cells:              {total_cells}  (manuscript: 26)")

assert total_obs == 3679, f"total_obs {total_obs} != manuscript 3679"
assert total_cells == 26, f"total_cells {total_cells} != manuscript 26"
print()
print("PASS: archive headline counts match the manuscript exactly.")

## $B^{\mathrm{eff}}_{\max}$ on the audit suite

Manuscript: $B^{\mathrm{eff}}_{\max} \approx 2.0$ on the audit, indicating the universal $B_5 = 227\,251$ is empirically loose by ~$10^5$ on this catalog/state suite.

In [ ]:
p = DATA / "screen_sector_audit_r5.json"
j = json.loads(p.read_text(encoding="utf-8"))

# Per-cell B_eff_max -> max over cells. The JSON schema lists cells under
# the "cells" key, each with a "B_eff_max" entry. We compute the
# max-over-cells explicitly rather than relying on a heuristic top-level key.
cells = j.get("cells") or j.get("rows") or []
per_cell = [c["B_eff_max"] for c in cells if "B_eff_max" in c]
assert per_cell, "no per-cell B_eff_max found in audit JSON"
val = max(per_cell)
print(f"B_eff_max (max over {len(per_cell)} cells in screen_sector_audit_r5.json): {val}")

# Manuscript Sec VI reports ~2.0; the deposited value is 2.0000125 (12.5 ppm
# above 2.0). The block-refined Theorem 3 predicts at most B_hat^charge_r
# in {1, 3, 5}, with 5 being the maximum on this catalog (Corollary 1), so the
# universal B_r = 105 is empirically loose by ~50x here.
assert 1.99 < val < 2.01, f"B_eff_max out of expected ~2.0 range: {val}"
print(f"PASS: B_eff_max in (1.99, 2.01); manuscript Sec VI reports ~2.0 (B_r=105 loose by ~50x).")

## Smoke audit on a tiny cell

Run the catalog evaluation pipeline on a single $n = 4$ Slater determinant. Section V proves $\Delta^{\mathrm{cat}}_{4,U(1)} = 0$ on every Slater determinant in the occupation basis, so we expect numerical zero for both $\Delta^{\mathrm{cat}}$ and $\max_W |\tau^G_W|$.

In [ ]:
from connected_layer_sector import determinant_state, evaluate_catalog
from connected_layer_sector.catalog import enumerate_chemistry_catalog

n_orb = 4
rho = determinant_state(n_orb, [1, 1, 0, 0])
metrics = evaluate_catalog(rho, n_orb, r=4)

# n_catalog_words is the number of CONCRETE word instances (5 chemistry-catalog
# word types from Sec III Def 3, instantiated at every distinct-site assignment
# allowed by the catalog's symmetry conventions on n_orb sites). At n_orb=4
# the package enumerates 47 such entries.
expected_n = len(enumerate_chemistry_catalog(n_orb))
assert metrics["n_catalog_words"] == expected_n, (
    f"n_catalog_words {metrics['n_catalog_words']} disagrees with "
    f"enumerate_chemistry_catalog length {expected_n}"
)

for k, v in metrics.items():
    print(f"  {k}: {v}")

# Sec V baseline: every cumulant of length 3 or 4 vanishes on a Slater
# determinant in the occupation basis.
assert metrics["delta_cat"] < 1e-12
assert metrics["max_tau"] < 1e-12
print()
print(f"PASS: Slater-determinant smoke audit reproduces Sec V zero baseline.")
print(f"PASS: n_catalog_words ({metrics['n_catalog_words']}) matches enumerate_chemistry_catalog at n_orb={n_orb}.")

## What's NOT here

The full $3679$-observable audit at $n = 8$ takes ~3 hours on a 32-worker laptop. The deposited JSONs in `data/` carry the headline numbers; `tests/test_data_integrity.py` verifies their SHA256 against `MANIFEST.json`. To re-run the full audit locally, see the original screens in the project working directory.